# Data Setup Phase

In this notebook, I'll isolate the data setup step. This makes it easier to inspect the DataFrame, verify file paths, and ensure everything is correct before moving on to feature extraction and modeling.

## Data Access & Reproducibility
**Note on Dataset Size:** The raw MP3 files and the generated intermediate acoustic matrices (Spectrograms, Model Checkpoints, and RMS arrays) total over **17 GB**. Due to GitHub and Canvas file size constraints, the raw audio and binary array files are excluded from this submission. 

### How to Reproduce from Scratch
If you wish to run this entire pipeline from the raw audio files:
1. Download the **FMA Small Dataset** (8 GiB) and the **FMA Metadata** (342 MiB) from the official [Free Music Archive (FMA) GitHub Repository](https://github.com/mdeff/fma).
2. Extract the audio into `data/fma_small/` and the metadata into `data/fma_metadata/`.
3. Run the `data_setup.ipynb` notebook to randomly sample the 200 tracks and generate the track paths.

### Running Without Raw Audio
To save you from downloading 17 GB of data, the final **extracted structural features** (e.g., optimal ARIMA parameters and PELT structural breaks) have been fully synthesized into lightweight CSV files. The final statistical inference blocks can be executed and verified instantaneously using these provided CSVs (such as `analysis_tracks_with_info.csv` and `method2_pelt_features.csv`) without needing to re-process the raw acoustic data.

In [1]:
import os
import numpy as np
import pandas as pd

# Define paths and parameters
metadata_path = 'data/fma_metadata/tracks.csv'
audio_dir = 'data/fma_small'
sample_size = 100
random_state = 42

## 1. Load and Filter Metadata
I need to load the FMA metadata, focus on the small and medium subsets, and find the Pop and Classical tracks.

In [2]:
# Load tracks metadata (header is on first two rows)
tracks = pd.read_csv(metadata_path, index_col=0, header=[0, 1])

# Filter for fma_small and fma_medium to ensure we include the downloaded classical tracks
valid_subsets = tracks[tracks[('set', 'subset')].isin(['small', 'medium'])]
genres = valid_subsets[('track', 'genre_top')]

# Get valid track IDs for our target genres
# Ensure Pop tracks are only from 'small' subset since we didn't download medium Pop tracks
small_subset = tracks[tracks[('set', 'subset')] == 'small']
pop_tracks = small_subset[small_subset[('track', 'genre_top')] == 'Pop'].index
classical_tracks = genres[genres == 'Classical'].index

print(f'Total available Pop tracks: {len(pop_tracks)}')
print(f'Total available Classical tracks: {len(classical_tracks)}')

Total available Pop tracks: 1000
Total available Classical tracks: 619


## 2. Sample Tracks
Now I sample exactly 100 tracks from each genre. I set the random seed to ensure reproducibility.

**Note:** I sample Classical *first* to match the exact sequence used by the downloader script.

In [3]:
# Set seed for reproducibility
np.random.seed(random_state)

# Sample Classical FIRST (this ensures we match the tracks downloaded by the downloader script)
classical_sample = np.random.choice(classical_tracks, sample_size, replace=False)

# Then sample Pop
pop_sample = np.random.choice(pop_tracks, sample_size, replace=False)

# Combine into a single list of indices
selected_indices = np.concatenate([pop_sample, classical_sample])

# Create the master DataFrame
df = pd.DataFrame({
    'track_id': selected_indices,
    'genre': genres.loc[selected_indices].values
})

display(df.head())
display(df['genre'].value_counts())

,track_id,genre
0,67360,Pop
1,127296,Pop
2,48453,Pop
3,86118,Pop
4,85427,Pop


genre
Pop          100
Classical    100
Name: count, dtype: int64

## 3. Construct and Verify Filepaths
Next, I generate the actual file paths pointing to the `.mp3` files in the `data/fma_small` folder, and verify that they all exist locally.

In [4]:
def get_filepath(track_id):
    # FMA structures folders by the first 3 digits of the padded 6-digit track ID
    tid_str = f'{track_id:06d}'
    folder_prefix = tid_str[:3]
    filename = f'{tid_str}.mp3'
    return os.path.join(audio_dir, folder_prefix, filename)

df['filepath'] = df['track_id'].apply(get_filepath)

# Verify files exist
df['file_exists'] = df['filepath'].apply(os.path.exists)

print(f'Total tracks: {len(df)}')
print(f"Tracks found on disk: {df['file_exists'].sum()}")

if df['file_exists'].sum() < len(df):
    missing = df[~df['file_exists']]
    print('\nWarning! Some tracks are missing from disk:')
    display(missing.head())
else:
    print('\nAll 200 audio files verified on disk! Ready for feature extraction.')

Total tracks: 200
Tracks found on disk: 200

All 200 audio files verified on disk! Ready for feature extraction.


## 4. Save Setup for Next Step
I can save this filtered DataFrame so I don't have to repeat this setup in the modeling notebook. I will load this CSV directly in step 1.

In [5]:
# Keep only existing files just in case (the 200 files we verified)
final_df = df[df['file_exists']].copy()

# Drop the helper column
final_df = final_df.drop(columns=['file_exists'])

# Save to CSV
final_df.to_csv('data/analysis_tracks.csv', index=False)
print('Saved final track index to data/analysis_tracks.csv')

Saved final track index to data/analysis_tracks.csv
